In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append(os.path.abspath('..'))  # on est dans notebooks/

In [2]:
import os, shutil, pathlib

# 1) (optionnel) étendre PATH si Tex live est dans /Library/TeX/texbin sur macOS
os.environ["PATH"] = "/Library/TeX/texbin:/usr/local/bin:/opt/homebrew/bin:" + os.environ.get("PATH", "")

# 2) config matplotlib cache pour logs LaTeX
os.environ["MPLCONFIGDIR"] = os.path.expanduser("~/mplconfig")
os.makedirs(os.path.join(os.environ["MPLCONFIGDIR"], "tex.cache"), exist_ok=True)

# 3) vérifier présence de latex/pdflatex
print("latex:", shutil.which("latex"))
print("pdflatex:", shutil.which("pdflatex"))

# 4) backend et rcParams (DOIT venir avant import pyplot)
import matplotlib as mpl
mpl.use("pgf")   # ou 'Agg' + text.usetex=True (mais pgf donne meilleur intégration)
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "pgf.rcfonts": False,
    "text.latex.preamble": r"\usepackage{siunitx}"
})

# 5) maintenant importer pyplot et générer la figure
import matplotlib.pyplot as plt
plt.plot([0,1,2],[0,1,0])
# Use simple math roman 's' instead of i to avoid backend PGF/siunitx catcode conflicts
plt.xlabel(r"Temps ($\mathrm{s}$)")
plt.tight_layout()
out = pathlib.Path.home() / "test_fig_pgf.pdf"
plt.savefig(out, bbox_inches="tight")
print("Wrote:", out)

# 6) logs LaTeX si besoin
tex_cache = pathlib.Path(os.environ["MPLCONFIGDIR"]) / "tex.cache"
logs = sorted(tex_cache.glob("*.log"), key=os.path.getmtime) if tex_cache.exists() else []
print("Dernier log:", logs[-1] if logs else "aucun")
if logs:
    print(logs[-1].read_text(errors="ignore").splitlines()[-40:])

latex: /Library/TeX/texbin/latex
pdflatex: /Library/TeX/texbin/pdflatex
Wrote: /Users/vl284796/test_fig_pgf.pdf
Dernier log: aucun


In [3]:
#pour les graphes en Latex
import os

# 1) (optionnel) étendre PATH si Tex live est dans /Library/TeX/texbin sur macOS
os.environ["PATH"] = "/Library/TeX/texbin:/usr/local/bin:/opt/homebrew/bin:" + os.environ.get("PATH", "")

# 2) config matplotlib cache pour logs LaTeX
os.environ["MPLCONFIGDIR"] = os.path.expanduser("~/mplconfig")
os.makedirs(os.path.join(os.environ["MPLCONFIGDIR"], "tex.cache"), exist_ok=True)

from pathlib import Path
import matplotlib as mpl

# --- Respecter la configuration faite dans la cellule 2 ---
# Ne pas changer le backend ici : la cellule 2 doit appeler mpl.use(...) avant
# Nous nous contentons d'assurer que MPLCONFIGDIR existe et de mettre à jour rcParams.
mplconfig = os.environ.get("MPLCONFIGDIR", os.path.expanduser("~/mplconfig"))
os.environ["MPLCONFIGDIR"] = mplconfig
os.makedirs(os.path.join(mplconfig, "tex.cache"), exist_ok=True)

# Optionnel : afficher le backend actuel pour debug
try:
    backend = mpl.get_backend()
except Exception:
    backend = 'unknown'
print("Matplotlib backend (cell 3):", backend)

# ——— Réglages LaTeX cohérents (sans toucher au backend) ———
# text.latex.preamble peut rester ; attention aux macros complexes (voir remarque siunitx)
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "axes.unicode_minus": False,
    "pgf.rcfonts": False,
    "text.latex.preamble": r"\usepackage{siunitx}",
    "font.size": 14,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "legend.fontsize": 12,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "lines.linewidth": 2.5,
})

# ——— Helper: taille en pouces à partir de \\textwidth ———
TEXTWIDTH_PT = 483.70  # ≈ 170 mm avec tes marges
def set_size(fraction=1.0, aspect=0.62, textwidth_pt=TEXTWIDTH_PT):
    """fraction de \\textwidth (0–1); aspect = hauteur/largeur."""
    inches_per_pt = 1/72.27
    w_in = textwidth_pt * inches_per_pt * fraction
    h_in = w_in * aspect
    return (w_in, h_in)

# ——— Dossier de sortie ———
out_dir = Path("/Users/vl284796/Documents/Images_for_Latex")
out_dir.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt


Matplotlib backend (cell 3): pgf


In [4]:
# Ajoute la racine du projet et importe les modules/fonctions utiles
import sys, os
from pathlib import Path


# import pratique - atm_tools
from src.cosmo_lidar.atm_tools import (
    effective_area_and_waist,
    alpha_specific_function,
    optical_depth_emission,
    contribution_effective_area,
    Calcul_T_ant_1_el,
    Calcul_T_ant_2,
    calcul_PWV,
    calcul_z_percentile_wvc,
    vapor_pressure,
    mass_quantile_grid,
    pwv_profile,
)

# import pratique - mc_tools
from src.cosmo_lidar.mc_tools import (
    generate_Pwater_MC,
    generate_Pwater_MC_lognormal,
    monte_carlo_t_ant,
    cumtrapz_like,
    sort_and_relative,
    build_features,
    snr_from_params,
    FitOptions,
    _initial_guess,
    fit_snr_combo,
    predict_snr_combo,
    _cumtrapz_like,
    compute_IR_IW,
    snr_from_params_2,
    calcul_snr,
    remove_nans,
    Monte_Carlo_T_ant_mod,
    local_bin_width,
    scale_snr_for_variable_bins,
    predict_SNR_T,
)

# import pratique - io
from src.cosmo_lidar.io import (
    fetch_html,
    extract_ut_column_dat_links,
    download_some,
    read_radiosonde_dat,
    read_many_radiosonde,
    save_table,
    load_table,
    load_parquet_columns_as_numpy,
    to_float64,
)

import numpy as np
pi = np.pi
from scipy.interpolate import UnivariateSpline
from scipy.integrate import simpson
from scipy.integrate import trapezoid, cumulative_trapezoid

ImportError: cannot import name 'cumtrapz_like' from 'src.cosmo_lidar.mc_tools' (/Users/vl284796/Documents/Cosmo-Lidar-Project/Cosmo-lidar-project/src/cosmo_lidar/mc_tools.py)

In [5]:
#Importation des données de Novembre

T_acq = 60 #temps d'acquisition (s)
E_0 = 60.e-3 #emission ernergy (J)
F_t = 200 #shooting frequency (Hz)

#pour un angle d'élévation de 90°

e=90 # angle d'élévation (°)
# Charger les données (adapter le nom du fichier et éventuellement le délimiteur)
data_2 = np.loadtxt("/Users/vl284796/Downloads/Simulation_Atakama_lidarH2O_50cm_00.txt")

# Chaque colonne devient un tableau séparé
s_90_2    = data_2[:, 0]  # distance au lidar (km)
z_90_2     = data_2[:, 1]  # altitude (km)
r_H2O_90_2 = data_2[:, 2] *1000 # WVMR (g/kg)
s_rh_90_2  = data_2[:, 3]  # écart type (g/kg)
b_rh_90_2 = data_2[:, 4]  # biais (g/kg)
p_90 = data_2[:,5] #Pression en hPa
T_90 = data_2[:,6] #Temperature (K)

frequency = np.array([150.e9]) #en Hz
theta_b = np.full(len(frequency), 1*60*pi/(2*180*3600)) #en rad
elev = 90 #en
N_MC = 100

In [ ]:
#On retrace les courbes

plt.figure(figsize=(6,5))

# Profil moyen avec barres d’erreurs
# Fill between (mean - std, mean + std)
plt.fill_betweenx(z_90_2,
                  r_H2O_90_2 - s_rh_90_2,
                  r_H2O_90_2 + s_rh_90_2,
                  color="gray", alpha=0.6, label="Standard deviation")

# Mean line
plt.plot(r_H2O_90_2, z_90_2, 'k-', lw=1.5, label="Mean value")

# Labels and styling
plt.xlabel("WVMR [g/kg]")
plt.ylabel("Altitude [km]")
#plt.title("Water Vapor Mixing Ratio with Uncertainty")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out_dir / "wvmr_90_2_with_uncertainty.pdf", bbox_inches="tight")

In [31]:
R_d_air = 287 #J/(kg*K)
R_water = 462 #J/(kg*K)


rho_water = p_90*100*r_H2O_90_2/(1000*R_d_air*T_90)*1/(1+R_water/R_d_air*r_H2O_90_2/1000) #en kg/m3
rho_air = p_90*100/(R_water*T_90) - rho_water*R_water/R_d_air

P_water = rho_water*R_water*T_90/100 

print(rho_water)

PWV = calcul_PWV(rho_water,z_90_2)

print(PWV)

[6.21162618e-04 6.08587804e-04 5.97482254e-04 5.87044692e-04
 5.76907119e-04 5.66907829e-04 5.56991811e-04 5.47141517e-04
 5.37351540e-04 5.27620097e-04 5.17946572e-04 5.08330569e-04
 4.98771906e-04 4.89270290e-04 4.79825529e-04 4.70437419e-04
 4.61105730e-04 4.51830383e-04 4.42612011e-04 4.33453413e-04
 4.24362801e-04 4.15357980e-04 4.06469802e-04 3.97742501e-04
 3.89229053e-04 3.80982571e-04 3.73046157e-04 3.65444853e-04
 3.58275028e-04 3.51331235e-04 3.44675593e-04 3.38266412e-04
 3.32059732e-04 3.26014379e-04 3.20095059e-04 3.14273540e-04
 3.08528554e-04 3.02844916e-04 2.97212291e-04 2.91624013e-04
 2.86076277e-04 2.80567500e-04 2.75098127e-04 2.69670641e-04
 2.64289814e-04 2.58962795e-04 2.53699138e-04 2.48510436e-04
 2.43409572e-04 2.38409709e-04 2.33523075e-04 2.28735113e-04
 2.24074047e-04 2.19545715e-04 2.15167423e-04 2.10906056e-04
 2.06771036e-04 2.02752806e-04 1.98839425e-04 1.95017662e-04
 1.91274154e-04 1.87596319e-04 1.83973093e-04 1.80395356e-04
 1.76856067e-04 1.733502

In [9]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Température
axs[0].plot(T_90, z_90_2, color="red")
axs[0].set_title("Temperature")
axs[0].set_ylabel("Altitudes (km)")
axs[0].set_xlabel("T (K)")
axs[0].grid(True)

# Pression
axs[1].plot(p_90, z_90_2, color="blue")
axs[1].set_title("Pressure")
axs[1].set_ylabel("Altitudes (km)")
axs[1].set_xlabel("P (hPa)")
axs[1].grid(True)

# Densité vapeur d'eau
axs[2].plot(P_water, z_90_2,color="green")
axs[2].set_title("Water vapor pressure")
axs[2].set_ylabel("Altitude (km)")
axs[2].set_xlabel(r'$P_{water}$ (hPa)')
axs[2].grid(True)

plt.tight_layout()
plt.savefig(out_dir / "t_p_pwater_lsce_simulation.pdf", bbox_inches="tight")

In [13]:
mu = r_H2O_90_2         # mean WVMR [g/kg]
s  = s_rh_90_2          # std [g/kg]
b  = b_rh_90_2          # bias [g/kg]

# avoid division by zero
eps = 1e-12
rmse = np.sqrt(s**2 + b**2)

SNR_rand    = mu / np.maximum(s, eps)
SNR_tot     = mu / np.maximum(rmse, eps)
SNR_rand_dB = 20*np.log10(SNR_rand)
SNR_tot_dB  = 20*np.log10(SNR_tot)

spline = UnivariateSpline(z_90_2[1:], SNR_rand[1:], s=1000)

snr_smooth_part = spline(z_90_2[1:])

# Réinsérer la première valeur brute
snr_smooth = np.insert(snr_smooth_part, 0, SNR_rand[0])


plt.figure(figsize=(6,5))
plt.plot(snr_smooth[1:], z_90_2[1:], lw=2.5)
plt.xlabel("SNR")
plt.xscale('log')
plt.ylabel("Altitude [km]")
#plt.title("WVMR SNR Profile")
plt.grid(alpha=0.3)
#plt.tight_layout()
plt.savefig(out_dir / "snr_wvmr_90_2_profile.pdf", bbox_inches="tight")

In [14]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))



axs[0].fill_betweenx(z_90_2,
                  r_H2O_90_2 - s_rh_90_2,
                  r_H2O_90_2 + s_rh_90_2,
                  color="gray", alpha=0.6, label="Standard deviation")

# Mean line
axs[0].plot(r_H2O_90_2, z_90_2, 'k-', lw=1.5, label="Mean value")

# Labels and styling
axs[0].set_xlabel("WVMR [g/kg]")
axs[0].set_ylabel("Altitude [km]")
axs[0].legend()
axs[0].grid(True)


axs[1].plot(snr_smooth[1:], z_90_2[1:], lw=2.5)
axs[1].set_xlabel("SNR")
axs[1].set_xscale('log')
axs[1].set_ylabel("Altitude [km]")
axs[1].grid(True)


plt.tight_layout()
plt.savefig(out_dir / "wvmr_snr_lsce_simulation.pdf", bbox_inches="tight")

In [15]:
A = 106533.049
c = 1.470303e-05
elev=90
snr_inch = calcul_snr (A, c, r_H2O_90_2, z_90_2*1000, p_90, T_90, elev)

In [16]:
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(snr_smooth[1:], z_90_2[1:], label='SNR from data')
#ax.plot(SNR_test, z_90_2)
ax.plot(snr_inch[1:], z_90_2[1:], label= 'SNR estimated')
#ax.plot(snr_inch_2, z_90_2[1:], label= 'SNR recalculé bis')
ax.set_xlabel("SNR (linear)")
ax.set_xscale("log")

ax.set_ylabel("Altitude [km]")
ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.savefig(out_dir / "snr_wvmr_90_2_fit_combo.pdf", bbox_inches="tight")

In [17]:
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(snr_smooth[1:]-snr_inch[1:], z_90_2[1:])
#ax.plot(SNR_test, z_90_2)
#ax.plot(snr_inch[1:], z_90_2[1:], label= 'SNR estimated')
#ax.plot(snr_inch_2, z_90_2[1:], label= 'SNR recalculé bis')
ax.set_xlabel("SNR simulated - SNR estimated (linear)")
ax.set_xscale("log")

ax.set_ylabel("Altitude [km]")
ax.grid(alpha=0.3); 
#ax.legend(); 
plt.tight_layout(); plt.savefig(out_dir / "snr_wvmr_90_2_fit_combo_dif.pdf", bbox_inches="tight")

In [19]:

N_MC = 1000
rel_sigma_WVMR_90 = 1/snr_smooth

r_H2O_90_MC = generate_Pwater_MC_lognormal(r_H2O_90_2, N_MC, rel_sigma_WVMR_90, rng=None)

WVMR0 = np.asarray(r_H2O_90_2, dtype=float)
WVMRmc = np.asarray(r_H2O_90_MC, dtype=float)

# Rapport (N_MC, n_alt)

ratio = np.where(WVMR0 == 0, WVMRmc/WVMR0, WVMRmc / WVMR0)
# Altitude en km pour affichage
z_km = z_90_2

# Moyenne et quantiles du rapport
ratio_mean = np.nanmean(ratio, axis=0)
ratio_p05  = np.nanpercentile(ratio, 5, axis=0)
ratio_p95  = np.nanpercentile(ratio, 95, axis=0)
ratio_p2_45  = np.nanpercentile(ratio, 2.45, axis=0)
ratio_p97_725  = np.nanpercentile(ratio, 97.725, axis=0)



# --- FIGURE 1 : profils Monte-Carlo + moyenne/IC ---
fig, ax = plt.subplots(figsize=(6, 5))

N_MC, n_alt = ratio.shape
rng = np.random.default_rng(42)
n_draws_plot = min(100, N_MC)
idx = rng.choice(N_MC, size=n_draws_plot, replace=False)

# Nuage de points (sous-échantillonné)
"""for i in idx:
    ax.scatter(ratio[i, :], z_km, s=5, color="grey", alpha=0.2)

# Moyenne et quantiles
ax.plot(ratio_mean, z_km, "k-", lw=2, label="Mean")
ax.fill_betweenx(z_km, ratio_p2_45, ratio_p97_725, color="skyblue", alpha=0.4, label="CI 95%")
ax.axvline(1.0, color="red", linestyle="--", label="Reference")
ax.plot(1 + 2/snr_smooth, z_km, color="green", label=r"1-2$\sigma$, 1+2$\sigma$")
ax.plot(1 - 2/snr_smooth, z_km, color="green")

# Mise en forme
ax.set_xlabel(r"$WVMR_{\mathrm{MC}} / WVMR_{\mathrm{ref}}$")
ax.set_ylabel("Altitude [km]")
ax.set_title("Monte-Carlo Verification : ratio $WVMR$")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(out_dir / "wvmr_90_2_mc_controle_SNR.pdf", bbox_inches="tight")"""
# plt.show()  # si tu veux afficher

# --- FIGURE 2 : histogramme à une altitude ---
i_alt = 150  # assure-toi que 0 <= i_alt < n_alt
fig2, ax2 = plt.subplots(figsize=(6, 4))

ax2.hist(r_H2O_90_MC[:, i_alt], bins=50, density=True, alpha=0.6, color="gray")
ax2.axvline(np.mean(r_H2O_90_MC[:, i_alt]), color="g", linestyle="-", label="mean")
ax2.axvline(r_H2O_90_2[i_alt], color="r", linestyle="--", label="Reference")

ax2.set_xlabel("WVMR (g/kg)")
ax2.set_ylabel("Density")
ax2.set_title(f"Distribution Monte Carlo (for z = {z_90_2[i_alt]} km)")
ax2.legend()

fig2.tight_layout()
fig2.savefig(out_dir / f"wvmr_90_2_mc_hist_alt_{z_90_2[i_alt]:.1f}km.pdf", bbox_inches="tight")
# plt.show()


<>:40: SyntaxWarning: invalid escape sequence '\s'
<>:40: SyntaxWarning: invalid escape sequence '\s'
/var/folders/z2/6hps10m55pd3mdx0n4vzfvy9w8t2v4/T/ipykernel_97318/2387189587.py:40: SyntaxWarning: invalid escape sequence '\s'
  ax.plot(1 + 2/snr_smooth, z_km, color="green", label=r"1-2$\sigma$, 1+2$\sigma$")


In [ ]:
from src.cosmo_lidar.mc_tools import monte_carlo_t_ant
N=500
N_MC=200
simu = monte_carlo_t_ant(frequency,theta_b, N , elev, generate_Pwater_MC_lognormal, N_MC , r_H2O_90_2, s_rh_90_2, T_90, p_90, z_90_2*1000)
print(simu, simu[0]/simu[1][0])

/Users/vl284796/Documents/Cosmo-Lidar-Project/Cosmo-lidar-project/src/cosmo_lidar/mc_tools.py:28: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  


(np.float64(0.0069400684381370255), array([5.4553948]))


In [30]:
print(simu, simu[0]/simu[1][0]*100)

(np.float64(0.0069400684381370255), array([5.4553948])) 0.12721477894095878


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.cosmo_lidar.mc_tools import monte_carlo_t_ant

# --- 1. Paramètres de simulation ---
N = 500
N_MC = 100

# Sauvegarde de la valeur de base de l'écart-type (votre "sigma_q")
sigma_q_base = s_rh_90_2 

# Définition de la plage de variation du facteur c
# Ex: on fait varier l'écart-type de 10% à 300% de sa valeur nominale
c_values = np.linspace(0.1, 3.0, 15) 

# Tableaux pour stocker les résultats
sigma_T_results = []
scaled_sigma_q_values = [] # Pour l'axe X (c * sigma_q)

print(f"Lancement de l'étude de sensibilité sur {len(c_values)} points...")

# --- 2. Boucle de calcul ---
for i, c in enumerate(c_values):
    
    # Calcul du nouveau paramètre s_rh pour cette itération
    current_s_rh = c * sigma_q_base
    
    # Appel de la fonction avec le paramètre modifié
    # Attention à bien passer current_s_rh à la place de s_rh_90_2
    result_tuple = monte_carlo_t_ant(
        frequency,
        theta_b, 
        N, 
        elev, 
        generate_Pwater_MC_lognormal, 
        N_MC, 
        r_H2O_90_2, 
        current_s_rh,  # <--- C'est ici que la variation s'applique
        T_90, 
        p_90, 
        z_90_2 * 1000
    )
    
    # Récupération de sigma_T (position 0 du tuple)
    val_sigma_T = result_tuple[0]
    
    # Stockage
    sigma_T_results.append(val_sigma_T)
    scaled_sigma_q_values.append(current_s_rh)
    
    #print(f"Iter {i+1}/{len(c_values)} : c={c:.2f} -> sigma_q={current_s_rh:.3f} => sigma_T={val_sigma_T:.4f} K")

# Conversion en arrays numpy pour faciliter le plot
sigma_T_results = np.array(sigma_T_results)
scaled_sigma_q_values = np.array(scaled_sigma_q_values)




<>:71: SyntaxWarning: invalid escape sequence '\s'
<>:71: SyntaxWarning: invalid escape sequence '\s'
/var/folders/z2/6hps10m55pd3mdx0n4vzfvy9w8t2v4/T/ipykernel_97318/3516483914.py:71: SyntaxWarning: invalid escape sequence '\s'
  plt.title(f"Sensibilité de $\sigma_T$ aux fluctuations d'humidité (N_MC={N_MC})")


Lancement de l'étude de sensibilité sur 15 points...


ValueError: x and y must be the same size

In [47]:
# --- 3. Visualisation ---
plt.figure(figsize=(10, 6))

# Nuage de points reliés
plt.scatter(c_values, sigma_T_results*1000, s=40, label='Monte-Carlo simulation')

# Régression linéaire pour vérifier la linéarité
# Si la relation est linéaire, sigma_T ~= k * sigma_q
coeffs = np.polyfit(c_values, sigma_T_results*1000, 1)
poly = np.poly1d(coeffs)
plt.plot(c_values, poly(c_values), 'r--', alpha=0.7, 
         label=f'linear fit : y = {coeffs[0]:.2f}x + {coeffs[1]:.2f}')

plt.xlabel(r"Factor of the standard deviation of q ($c \cdot \sigma_q$)")
plt.ylabel(r"Standard deviation on the antenna temperature $\sigma_T$ [mK]")
#plt.title(r"Sensibility of $\sigma_T$ with variations of $\sigma_q$($N_{MC}$=100)")
plt.legend()
plt.grid(True)
plt.savefig(out_dir / "variation_sigma_q_sigma_t_impact.pdf", bbox_inches="tight")

In [58]:
N_MC = 200
# On définit l'espace des élévations : 10 valeurs entre 30 et 90 degrés
elevations_vals = np.linspace(25, 90, 25)

# Tableaux pour stocker les résultats
sigma_T_elev_results = []

print(f"Lancement de l'étude sur l'élévation ({len(elevations_vals)} points)...")

# --- 2. Boucle de calcul ---
for i, current_elev in enumerate(elevations_vals):
    
    # Appel de la fonction avec l'élévation courante
    # On garde s_rh_90_2 constant (valeur de base)
    result_tuple = monte_carlo_t_ant(
        frequency,
        theta_b, 
        N, 
        current_elev,   # <--- Paramètre variable ici
        generate_Pwater_MC_lognormal, 
        N_MC, 
        r_H2O_90_2, 
        s_rh_90_2,      # Paramètre constant ici
        T_90, 
        p_90, 
        z_90_2 * 1000
    )
    
    # Récupération de sigma_T (position 0 du tuple)
    val_sigma_T = result_tuple[0]
    
    # Stockage
    sigma_T_elev_results.append(val_sigma_T)
    
    #print(f"Iter {i+1}/{len(elevations_vals)} : Elev={current_elev:.1f}° => sigma_T={val_sigma_T:.4f} K")

# Conversion en array numpy
sigma_T_elev_results = np.array(sigma_T_elev_results)


Lancement de l'étude sur l'élévation (25 points)...


In [59]:
# --- 3. Visualisation ---
pi = np.pi
thet_vals = 90 - elevations_vals
thet_rad_vals= thet_vals * pi/180
m = 1/(np.cos(thet_rad_vals) + 0.50572*(96.07995-thet_vals)**(-1.6364))
plt.figure(figsize=(10, 6))

# Nuage de points reliés
plt.scatter(m, sigma_T_elev_results*1000, s=40, label='Monte-Carlo simulation')



# Régression linéaire pour vérifier la linéarité
# Si la relation est linéaire, sigma_T ~= k * sigma_q
coeffs = np.polyfit(m, sigma_T_elev_results*1000, 1)
poly = np.poly1d(coeffs)
plt.plot(m, poly(m), 'r--', alpha=0.7, 
         label=f'linear fit : y = {coeffs[0]:.2f}x + {coeffs[1]:.2f}')

plt.xlabel("airmass m(el)")
plt.ylabel(r"Standard deviation on $\sigma_T$ [mK]")
#plt.title(r"Sensibility of $\sigma_T$ with variations of $\sigma_q$($N_{MC}$=100)")
plt.legend()
plt.grid(True)
plt.savefig(out_dir / "variation_elevation_angle_sigma_t_impact.pdf", bbox_inches="tight")

In [56]:
print(elevations_vals,m)

[30.         33.15789474 36.31578947 39.47368421 42.63157895 45.78947368
 48.94736842 52.10526316 55.26315789 58.42105263 61.57894737 64.73684211
 67.89473684 71.05263158 74.21052632 77.36842105 80.52631579 83.68421053
 86.84210526 90.        ] [1.99429285 1.82416688 1.68539092 1.57059651 1.47458458 1.39358737
 1.32481251 1.26615118 1.21598558 1.17305848 1.1363828  1.1051778
 1.07882315 1.05682557 1.03879432 1.02442305 1.01347646 1.00578055
 1.00121567 0.99971199]


In [17]:
import pandas as pd
import matplotlib.pyplot as plt
# Lire le fichier .txt


cols = ["TIME", "PMB", "TEMP", "TDEW", "RH", 
        "GEOP", "AZ", "EL", "SPEED", "DIR", 
        "E.TIME", "RT", "Battery"]

df_1 = pd.read_csv('/Users/vl284796/Downloads/data 25 avril.txt', 
                 delim_whitespace=True, 
                 skiprows=1, 
                 names=cols, encoding='latin-1')

time = df_1["TIME"].values
temp = pd.to_numeric(df_1["TEMP"], errors="coerce").values[2:-1780] + 273.15 #en K
alt = pd.to_numeric(df_1["GEOP"], errors="coerce").values[2:-1780] *1.e-3 #en km
rh   = pd.to_numeric(df_1["RH"], errors="coerce").values[2:-1780] #%
T_DEW = pd.to_numeric(df_1["TDEW"], errors="coerce").values[2:-1780] +273.15 # en K
P_data =  pd.to_numeric(df_1["PMB"], errors="coerce").values[2:-1780] #en hPa



e_1= vapor_pressure(T_DEW) #hPa
r_WVMR_1 = 0.622*e_1/(P_data-e_1) #kg/kg

/var/folders/z2/6hps10m55pd3mdx0n4vzfvy9w8t2v4/T/ipykernel_54389/2512275130.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df_1 = pd.read_csv('/Users/vl284796/Downloads/data 25 avril.txt',


In [18]:
plt.figure(figsize=(7,5))
plt.plot(P_data, alt, label="NRAO data", color='royalblue', linewidth=2)
plt.plot(p_90, z_90_2, label="standardized data", color='orange', linewidth=2)


# Mise en forme
plt.xlabel("Pressure (hPa)")
plt.ylabel("Altitudes (km)")
plt.title("P(z)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out_dir / "pressure_profile_comparison_nrao_lsce.pdf", bbox_inches="tight")


plt.figure(figsize=(7,5))
plt.plot(temp, alt, label="NRAO data", color='royalblue', linewidth=2)
plt.plot(T_90, z_90_2, label="standardized data", color='orange', linewidth=2)


# Mise en forme
plt.xlabel("Temperature (K)")
plt.ylabel("Altitudes (km)")
plt.title("T(z)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out_dir / "temperature_profile_comparison_nrao_lsce.pdf", bbox_inches="tight")

In [19]:
WVMR = r_WVMR_1 #g/kg
rho_water_1 = P_data*100*WVMR/(R_d_air*temp)*1/(1+R_water/R_d_air*WVMR) #en kg/m3

integrale_cum = cumulative_trapezoid(rho_water_1[1:], alt[1:]*1000, initial=0)

# Valeur totale de l’intégrale
I_tot = integrale_cum[-1]

# Pour chaque z_i : intégrale de z_i à z_max
I_from_z = I_tot - integrale_cum

integrale_cum = cumulative_trapezoid(rho_water, z_90_2*1000, initial=0)

# Valeur totale de l’intégrale
I_tot = integrale_cum[-1]

# Pour chaque z_i : intégrale de z_i à z_max
I_from_z_90 = I_tot - integrale_cum

# Tracé
plt.figure(figsize=(7,5))
plt.plot(I_from_z, alt[1:], label=r"$I(z)=\int_z^{z_{max}} \rho(z')\,dz'$ from NRAO")
plt.plot(I_from_z_90, z_90_2, label=r"$I(z)=\int_z^{z_{max}} \rho(z')\,dz'$ standardized")
#plt.plot(I_from_z_45, z_45_2, label=r"$I(z)=\int_z^{z_{max}} \rho(z')\,dz'$ 45")

plt.xlabel("water vapor column (mm)")
plt.ylabel("altitudes (km)")
#plt.ylim(top = 10)
plt.legend()
plt.grid(True)
plt.savefig(out_dir / "water_vapor_column_profile_comparison_nrao_lsce.pdf", bbox_inches="tight")


In [20]:
z = alt[1:]
WVMR = r_WVMR_1[1:]*1000


fig, ax = plt.subplots(figsize=(7,5))
ax.plot(WVMR, z, label='WVMR from NRAO')
ax.plot(r_H2O_90_2, z_90_2, label='standardized WVMR')
#ax.plot(SNR_test, z_90_2)
#ax.plot(pred_2, z_45_2, label= 'SNR recalculé')
ax.set_xlabel("WVMR (g/kg)")
#ax.set_xscale("log")

ax.set_ylabel("Altitude [km]")
ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.savefig(out_dir / "wvmr_profile_comparison_nrao_lsce.pdf", bbox_inches="tight")

Calcul du début pour estimer la température de l'antenne

In [5]:
pi = np.pi     
frequency = np.linspace(100.e9 , 600.e9 , 200) #Hz
wavelength = 3.e8 / frequency #m
theta_b = np.full(len(frequency), 1*60*pi/(2*180*3600))
w_0 = wavelength / (pi*theta_b) #m
altitudes = np.geomspace(1, 5001, 1001) #m
altitudes = altitudes +4999 #m

print(altitudes)

[ 5000.          5000.00855377  5000.01718071 ...  9915.53053577
  9957.58540406 10000.        ]


In [46]:
import pycraf
from pycraf import conversions as cnv
from astropy import units as u


# Conversion en Quantity
altitudes_km = altitudes * u.m       # maintenant c'est une Quantity en m
altitudes_km = altitudes_km.to(u.km) # conversion en km
frequency_GHz = frequency*10**-9 * u.GHz



profile_standard = pycraf.atm.profile_standard(altitudes_km)
Temperature = pycraf.atm.profile_standard(altitudes_km)[0] #en K
Pressure = pycraf.atm.profile_standard(altitudes_km)[1] #en hPa
rho_water = pycraf.atm.profile_standard(altitudes_km)[2] #en g/m3
P_water = pycraf.atm.profile_standard(altitudes_km)[3] #en hPa
P_dry = Pressure - P_water #en hPa

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Température
axs[0].plot(Temperature.value, altitudes/1000, color="red")
axs[0].set_title("Temperature")
axs[0].set_ylabel("Altitudes (km)")
axs[0].set_xlabel("T (K)")
axs[0].grid(True)

# Pression
axs[1].plot(Pressure.value, altitudes/1000, color="blue")
axs[1].set_title("Pressure")
axs[1].set_ylabel("Altitudes (km)")
axs[1].set_xlabel("P (hPa)")
axs[1].grid(True)

# Densité vapeur d'eau
axs[2].plot(P_water.value,altitudes/1000 ,color="green")
axs[2].set_title("Water vapor pressure")
axs[2].set_ylabel("Altitude (km)")
axs[2].set_xlabel(r'$P_{water}$ (hPa)')
axs[2].grid(True)

plt.tight_layout()
plt.savefig(out_dir / "t_p_pwater_pycraf.pdf", bbox_inches="tight")


ValueError: Error measuring {\rmfamily\fontsize{14.000000}{16.800000}\selectfont\catcode`\^=\active\def^{\ifmmode\sp\else\^{}\fi}\catcode`\%=\active\def%{\%}$P_{water} (hPa)}
LaTeX Output:
! Extra }, or forgotten $.
<argument> ...=\active \def %{\%}$P_{water} (hPa)}
                                                  
<*> ...tcode`\%=\active\def%{\%}$P_{water} (hPa)}}
                                                  \typeout{\the\wd0,\the\ht0...

!  ==> Fatal error occurred, no output PDF file produced!
Transcript written on texput.log.


In [38]:
iwc = pwv_profile(rho_water.value/1000, altitudes)
plt.figure(figsize=(8,5))
plt.plot(iwc, altitudes/1000, color="purple")
#plt.title("Integrated water vapor content profile")
plt.xlabel("IWV (kg/m²)")
plt.ylabel("Altitude (km)")
plt.grid(True)
plt.tight_layout()
plt.savefig(out_dir / "iwc_profile_pycraf.pdf", bbox_inches="tight")

In [41]:
print(iwc[0]) #en mm

1.1302125846876663


In [7]:
frequencies = np.array([30e9, 90e9, 150e9, 220e9])
altitudes = np.geomspace(1, 5001, 1001) #m
altitudes = altitudes +4999 #m

alpha = alpha_specific_function(altitudes, frequencies, Temperature.value, Pressure.value, P_water.value)

plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    
    plt.plot(alpha[:,i]*1000, altitudes/1000, label=f"{frequencies[i]*1.e-9:.1f} GHz")
plt.xlabel("Specific power attenuation (km-1)")
#plt.xscale("log")
plt.ylabel("Altitude (km)")
#plt.title("Specific attenuation profiles at various frequencies")
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "specific_attenuation_profiles_pycraf.pdf", bbox_inches="tight")


In [8]:
plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    
    plt.plot(alpha[:,i]*1000/P_water.value, altitudes/1000, label=f"{frequencies[i]*1.e-9:.1f} GHz")
plt.xlabel(r"$\alpha$/$P_w$ ((hPa.km)-1)")
#plt.xscale("log")
plt.ylabel("Altitude (km)")
#plt.title(r"$\alpha$/$P_w$ at various frequencies")
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "specific_attenuation_profiles_pycraf_divid_pwater.pdf", bbox_inches="tight")

In [9]:
plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    
    plt.plot(alpha[:,i]*1000/Pressure.value, altitudes/1000, label=f"{frequencies[i]*1.e-9:.1f} GHz")
plt.xlabel("Specific power attenuation (km-1)")
#plt.xscale("log")
plt.ylabel("Altitude (km)")
#plt.title("Specific attenuation profiles at various frequencies")
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "specific_attenuation_profiles_pycraf_divid_pressure.pdf", bbox_inches="tight")

In [8]:
tau = optical_depth_emission(altitudes, alpha)

plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    
    plt.plot(tau[:,i], altitudes/1000, label=f"{frequencies[i]*1.e-9:.1f} GHz")
plt.xlabel("Optical depth (unitless)")
#plt.xscale("log")
plt.ylabel("Altitude (km)")
#plt.title("Optical depth at various frequencies")
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "optical_depth_profile_pycraf_f.pdf", bbox_inches="tight")

In [9]:
total = alpha * np.exp(-tau)
print(total.shape)


plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    
    plt.plot(total[:,i], altitudes/1000, label=f"{frequencies[i]*1.e-9:.1f} GHz")
plt.xlabel("alpha*exp(-tau) (m-1)")
#plt.xscale("log")
plt.ylabel("Altitude (km)")
#plt.title("total at various frequencies")
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "total_profile_pycraf_f.pdf", bbox_inches="tight")

(1001, 4)


In [14]:
plt.figure(figsize=(8,5))
plt.plot(total[:,2]*1000, altitudes/1000, linestyle = '-',label=r'$\alpha e^{-\tau}$ at 150 GHz')
plt.plot(alpha[:,2]*1000, altitudes/1000,linestyle = '--', label=r'$\alpha$ at 150 GHz')
plt.xlabel(r'$\alpha e^{-\tau}$ and $\alpha$ (km$^{-1}$)')
plt.ylabel("Altitude (km)")
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "total_vs_alpha_profile_pycraf_150ghz.pdf", bbox_inches="tight")


In [74]:
plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    
    plt.plot(alpha[:,i], total[:,i], label=f"{frequencies[i]*1.e-9:.1f} GHz")
plt.xlabel("alpha (m-1)")
#plt.xscale("log")
plt.ylabel("total (m-1)")
#plt.title("total at various frequencies")
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "alpha_vs_total.pdf", bbox_inches="tight")

In [75]:
plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    
    plt.plot(alpha[:,i], np.exp(-tau[:,i]), label=f"{frequencies[i]*1.e-9:.1f} GHz")
plt.xlabel("alpha (m-1)")
#plt.xscale("log")
plt.ylabel("np.exp(-tau) (m)")
#plt.title("total at various frequencies")
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "alpha_vs_exp-tau.pdf", bbox_inches="tight")

In [ ]:
#calculons  le terme que l'on va intégrer selon z

#theta_b = np.full(len(frequencies), 1*60*pi/(2*180*3600))
elev = 90

C_tel = contribution_effective_area(frequencies, theta_b, altitudes, elev)

#C_tot = total * C_tel * Temperature

plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    C_tot = total[:,i]*C_tel[:,i]*Temperature
    plt.plot(altitudes/1000, C_tot*1000, label=f"{frequencies[i]*1.e-9:.1f} GHz")
plt.xlabel("altitudes (km)")
#plt.xscale("log")
plt.ylabel("C_tot (K/km)")
#plt.title("total at various frequencies")
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "C_tot_various_frequencies.pdf", bbox_inches="tight")

NameError: name 'frequencies' is not defined

In [88]:
#on va calculer la temperature d'antenne
pi = np.pi     
frequency = np.linspace(100.e9 , 600.e9 , 800) #Hz
wavelength = 3.e8 / frequency #m
theta_b = np.full(len(frequency), 1*60*pi/(2*180*3600))
elevation = 90
altitudes = np.geomspace(1, 5001, 1000) #m
altitudes = altitudes +4999 #m

altitudes_km = altitudes * u.m       # maintenant c'est une Quantity en m
altitudes_km = altitudes_km.to(u.km) # conversion en km
Temperature = pycraf.atm.profile_standard(altitudes_km)[0] #en K
Pressure = pycraf.atm.profile_standard(altitudes_km)[1] #en hPa
rho_water = pycraf.atm.profile_standard(altitudes_km)[2] #en g/m3
P_water = pycraf.atm.profile_standard(altitudes_km)[3] #en hPa
P_dry = Pressure - P_water #en hPa

T_ant_90 = Calcul_T_ant_1_el(frequency,theta_b, altitudes, Temperature.value, Pressure.value, P_water.value, elevation)

In [37]:
plt.figure(figsize=(8,5))
plt.plot(frequency*1.e-9, T_ant_90, marker='o', markersize=5)
plt.ylabel(r'$T_{ant}$ (K)')
plt.xlabel("Frequency (GHz)")
#plt.title("Antenna temperature at 90° elevation")
plt.grid(True)
plt.tight_layout()
plt.savefig(out_dir / "T_ant_90deg.pdf", bbox_inches="tight")

In [ ]:
plt.figure(figsize=(8,5))          # <-- nouvelle figure (évite de réutiliser l’ancienne)
# ou: plt.clf() avant de tracer si tu préfères

elevations = [90, 60, 30, 15]
T_ant_all = {}

for elev in elevations:
    T_ant = Calcul_T_ant_1_el(frequency, theta_b, altitudes,
                              Temperature.value, Pressure.value, P_water.value, elev)
    T_ant_all[elev] = T_ant
    plt.plot(frequency*1e-9, T_ant,
             #marker='o', markersize=5,
             label=fr"${elev}^\circ$")   # <-- pas de ° unicode

plt.ylabel(r'$T_{ant}$ (K)')
plt.xlabel("Frequency (GHz)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(out_dir / "T_ant_various_elevations.pdf", bbox_inches="tight")

In [42]:
plt.figure(figsize=(8,5))
factor = [0.5, 1, 2, 4]
PWV = 1.13
for f in factor:
    P_water_mod = P_water.value * f
    T_ant = Calcul_T_ant_1_el(frequency, theta_b, altitudes,
                              Temperature.value, Pressure.value, P_water_mod, 90)
    plt.plot(frequency*1e-9, T_ant,
             label=fr"PWV x {f:.1f}")
plt.ylabel(r'$T_{ant}$ (K)')
plt.xlabel("Frequency (GHz)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(out_dir / "T_ant_various_PWV.pdf", bbox_inches="tight")

In [43]:
f=np.array([150.e9]) #Hz
theta_b_f = np.full(len(f), 1*60*pi/(2*180*3600))
elevation_f = 90
PWV = 1.13
x = np.linspace(0.1, 9, 100)
PWV_x = x * PWV
T_ant_test = np.zeros_like(x)
i=0
for val in x :
    
    P_water_test = P_water * (val)
    T_ant_test[i] = Calcul_T_ant_1_el(f,theta_b_f, altitudes, Temperature.value, Pressure.value, P_water_test.value, elevation_f)
    i=i+1

/var/folders/z2/6hps10m55pd3mdx0n4vzfvy9w8t2v4/T/ipykernel_2729/1653497269.py:12: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  T_ant_test[i] = Calcul_T_ant_1_el(f,theta_b_f, altitudes, Temperature.value, Pressure.value, P_water_test.value, elevation_f)


In [45]:
plt.figure(figsize=(8,5))
plt.plot(PWV_x, T_ant_test/PWV_x)
plt.xlabel("PWV (mm)")
plt.xlim(0.1, 10)
plt.ylim(4,10)
plt.ylabel(r'$T_{ant}$/PWV (K/mm)')
#plt.title("Antenna temperature at 150 GHz vs PWV")
plt.grid(True)
plt.tight_layout()
plt.savefig(out_dir / "T_ant_150GHz_vs_PWV.pdf", bbox_inches="tight")

In [48]:


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

# --- Gauche : T_ant en fonction de PWV ---
ax1.plot(PWV_x, T_ant_test)
ax1.set_xlabel("PWV (mm)")
ax1.set_ylabel(r"$T_{\mathrm{ant}}$ (K)")
ax1.grid(True)
ax1.set_xlim(0, 10)
#ax1.set_title(r"$T_{\mathrm{ant}}$ vs PWV")

# --- Droite : T_ant / PWV en fonction de PWV ---
# (option de sécurité si des PWV==0)
ratio = T_ant_test / np.where(PWV_x == 0, np.nan, PWV_x)
ax2.plot(PWV_x, ratio)
ax2.set_xlabel("PWV (mm)")
ax2.set_ylabel(r"$T_{\mathrm{ant}}/\mathrm{PWV}$ (K/mm)")
ax2.set_xlim(0, 10)
ax2.set_ylim(4, 8)
ax2.grid(True)
#ax2.set_title(r"$T_{\mathrm{ant}}/\mathrm{PWV}$ vs PWV")

fig.tight_layout()
fig.savefig(out_dir / "T_ant_150GHz_vs_PWV_combined.pdf", bbox_inches="tight")


essayons d'enlever la contribution du dry air

In [100]:
f=np.array([150.e9]) #Hz
altitudes = np.geomspace(1, 15001, 1000) #m
altitudes = altitudes +4999 #m

altitudes_km = altitudes * u.m       # maintenant c'est une Quantity en m
altitudes_km = altitudes_km.to(u.km) # conversion en km
Temperature = pycraf.atm.profile_standard(altitudes_km)[0] #en K
Pressure = pycraf.atm.profile_standard(altitudes_km)[1] #en hPa
rho_water = pycraf.atm.profile_standard(altitudes_km)[2] #en g/m3
P_water = pycraf.atm.profile_standard(altitudes_km)[3] #en hPa
P_dry = Pressure - P_water #en hPa
theta_b_f = np.full(len(f), 1*60*pi/(2*180*3600))
elevation_f = 90
PWV = 1.13
x = np.linspace(0.1, 9, 100)
PWV_x = x * PWV
T_ant_test = np.zeros_like(x)
T_ant_dry_air = np.zeros_like(x)
i=0
P_water_test_wo_water = P_water * 10e-15
T_ant_dry = Calcul_T_ant_1_el(f,theta_b_f, altitudes, Temperature.value, Pressure.value, P_water_test_wo_water.value, elevation_f)
for val in x :
    
    P_water_test = P_water * (val)
    P_water_test_wo_water = P_water * (val) * 10e-15
    T_ant_test[i] = Calcul_T_ant_1_el(f,theta_b_f, altitudes, Temperature.value, Pressure.value, P_water_test.value, elevation_f)
    T_ant_dry_air[i] = T_ant_dry
    
    i=i+1
    
T_ant_only_water = T_ant_test - T_ant_dry_air    

/var/folders/z2/6hps10m55pd3mdx0n4vzfvy9w8t2v4/T/ipykernel_97318/4014332004.py:26: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  T_ant_test[i] = Calcul_T_ant_1_el(f,theta_b_f, altitudes, Temperature.value, Pressure.value, P_water_test.value, elevation_f)
/var/folders/z2/6hps10m55pd3mdx0n4vzfvy9w8t2v4/T/ipykernel_97318/4014332004.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  T_ant_dry_air[i] = T_ant_dry


In [95]:
print(T_ant_dry_air, T_ant_test)

[1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213
 1.20044213 1.20044213 1.20044213 1.20044213 1.20044213 1.2004

In [103]:

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

# --- Gauche : T_ant en fonction de PWV ---
ax1.plot(PWV_x, T_ant_only_water)
ax1.set_xlabel("PWV (mm)")
ax1.set_ylabel(r"$T_{\mathrm{ant}}$ [K] (water only)")
ax1.grid(True)
ax1.set_xlim(0, 10)
#ax1.set_title(r"$T_{\mathrm{ant}}$ vs PWV")

# --- Droite : T_ant / PWV en fonction de PWV ---
# (option de sécurité si des PWV==0)
ratio = T_ant_only_water / np.where(PWV_x == 0, np.nan, PWV_x)
ax2.plot(PWV_x, ratio)
ax2.set_xlabel("PWV (mm)")
ax2.set_ylabel(r"$T_{\mathrm{ant}}/\mathrm{PWV}$ [K/mm] (water only)")
ax2.set_xlim(0, 10)
#ax2.set_ylim(4, 8)
ax2.grid(True)
#ax2.set_title(r"$T_{\mathrm{ant}}/\mathrm{PWV}$ vs PWV")

fig.tight_layout()
fig.savefig(out_dir / "T_ant_only_water_150GHz_vs_PWV_combined.pdf", bbox_inches="tight")

In [49]:
f=np.array([150.e9]) #Hz
theta_b_f = np.full(len(f), 1*60*pi/(2*180*3600))
elevation_f = 90

x = np.linspace(0.22, 1, 50)
elevation_x = x *  elevation_f
T_ant_test_2 = np.zeros_like(x)
i=0
for val in elevation_x :
    
    
    T_ant_test_2[i] = Calcul_T_ant_1_el(f,theta_b_f, altitudes, Temperature.value, Pressure.value, P_water.value, val)
    i=i+1

/var/folders/z2/6hps10m55pd3mdx0n4vzfvy9w8t2v4/T/ipykernel_2729/1289040848.py:12: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  T_ant_test_2[i] = Calcul_T_ant_1_el(f,theta_b_f, altitudes, Temperature.value, Pressure.value, P_water.value, val)


In [50]:
plt.figure(figsize=(8,5))
plt.plot(elevation_x, T_ant_test_2*np.sin(elevation_x*pi/180))
plt.xlabel(r'elevation angle ($^\circ$)')
#plt.xlim(0.5, 12)
plt.ylim(6,7)
plt.ylabel(r'$T_{ant} \cdot \sin(elevation)$ (K)')
#plt.title("Antenna temperature at 150 GHz vs PWV")
plt.grid(True)
plt.tight_layout()
plt.savefig(out_dir / "T_ant_150GHz_vs_elevation.pdf", bbox_inches="tight")

Tracer des premieres courbes

In [ ]:
pi = np.pi
altitudes = np.arange(0.001, 100, 0.1)   # m
elevation_90 = 90                         # deg
# N supposé connu dans ton code :
# N = ...

# ==============================
# 1) Fréquence FIXE, plusieurs θ_b
# ==============================

# Exemple d’ouvertures : 17", 1', 9' (en radians)
arcsec = pi / (180 * 3600)
arcmin = 60 * arcsec
theta_b_values = np.array([1*arcsec/2,17 * arcsec/2, 60 * arcsec/2, 500 * arcsec/2,3000 * arcsec/2])

frequency_fixed = np.full(len(theta_b_values), 150.e9)

N=500

C_tel = contribution_effective_area(frequency_fixed, theta_b_values, altitudes, elevation_90, N)
i=0
plt.figure(figsize=(7,5))
for theta_b in theta_b_values:
    
    label = fr"$\theta_b$ = {theta_b/arcsec:.3g}''"  # affiche en arcmin
    plt.plot(altitudes, C_tel[:,i], label=label)
    i+=1

plt.xlabel('Altitude (m)')
plt.ylabel(r'$C_{\rm tel}$')
plt.title(r'Impact of the telescope aperture on $C_{tel}$ (f = 150 GHz)')
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "C_tel_vs_altitude_theta_b_variation.pdf", bbox_inches="tight")

# ==============================
# 2) θ_b FIXE, plusieurs fréquences
# ==============================

frequencies = np.array([30e9, 90e9, 150e9, 300e9, 480e9])  # Hz (exemples usuels)

theta_b_fixed = np.full(len(frequencies), 1*60*pi/(2*180*3600))
C_tel = contribution_effective_area(frequencies, theta_b_fixed, altitudes, elevation_90, N)
j=0
plt.figure(figsize=(7,5))
for f in frequencies:
    
    plt.plot(altitudes, C_tel[:,j], label=f"{f/1e9:.0f} GHz")
    j+=1

plt.xlabel('Altitude (m)')
plt.ylabel(r'$C_{\rm tel}$')
plt.title(r"Impact of the frequency on $C_{\rm tel}$ ($\theta_{b}$ = 0.5'') ")
plt.legend(title='Fréquence')
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "C_tel_vs_altitude_frequency_variation.pdf", bbox_inches="tight")

Comparaison pour l'influence de PWV et T sur alpha

In [14]:
import pycraf
from pycraf import conversions as cnv
from astropy import units as u


pi = np.pi     
frequency = np.linspace(100.e9 , 600.e9 , 200) #Hz
wavelength = 3.e8 / frequency #m
theta_b = np.full(len(frequency), 1*60*pi/(2*180*3600))
w_0 = wavelength / (pi*theta_b) #m
altitudes = np.geomspace(1, 5001, 1001) #m
altitudes = altitudes +4999 #m

# Conversion en Quantity
altitudes_km = altitudes * u.m       # maintenant c'est une Quantity en m
altitudes_km = altitudes_km.to(u.km) # conversion en km
frequency_GHz = frequency*10**-9 * u.GHz



profile_standard = pycraf.atm.profile_standard(altitudes_km)
Temperature = pycraf.atm.profile_standard(altitudes_km)[0] #en K
Pressure = pycraf.atm.profile_standard(altitudes_km)[1] #en hPa
rho_water = pycraf.atm.profile_standard(altitudes_km)[2] #en g/m3
P_water = pycraf.atm.profile_standard(altitudes_km)[3] #en hPa
P_dry = Pressure - P_water #en hPa

In [15]:
frequencies = np.array([30e9, 90e9, 150e9, 220e9])
#altitudes = np.geomspace(1, 5001, 1001) #m
#altitudes = altitudes +4999 #m

alpha = alpha_specific_function(altitudes, frequencies, Temperature.value, Pressure.value, P_water.value)

P_water_plus = 5 * P_water 

Temperature_plus = 2 * Temperature

alpha_P_w_plus = alpha_specific_function(altitudes, frequencies, Temperature.value, Pressure.value, P_water_plus.value)

alpha_T_plus = alpha_specific_function(altitudes, frequencies, Temperature_plus.value, Pressure.value, P_water.value)

In [18]:
plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    
    plt.plot(alpha[:,i]*1000, altitudes/1000, label=f"{frequencies[i]*1.e-9:.1f} GHz")
    plt.plot(alpha_T_plus[:,i]*1000, altitudes/1000, linestyle ='--', label=f"{frequencies[i]*1.e-9:.1f} GHz" )
plt.xlabel("Specific power attenuation (km-1)")

plt.ylabel("Altitude (km)")

plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "specific_attenuation_profiles_pycraf_2_T_comp.pdf", bbox_inches="tight")

In [22]:
plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    
    plt.plot(alpha[:,i]*1000/P_water.value, altitudes/1000, label=f"{frequencies[i]*1.e-9:.1f} GHz")
    plt.plot(alpha_P_w_plus[:,i]*1000/P_water_plus.value, altitudes/1000, linestyle ='--', label=f"{frequencies[i]*1.e-9:.1f} GHz" )
plt.xlabel("Specific power attenuation (km-1)")

plt.ylabel("Altitude (km)")

plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "specific_attenuation_profiles_pycraf_5_P_w_comp.pdf", bbox_inches="tight")

In [ ]:
plt.figure(figsize=(8,5))
for i in range (len(frequencies)):
    
    plt.plot(alpha[:,i]*1000, altitudes/1000, label=f"{frequencies[i]*1.e-9:.1f} GHz")
plt.xlabel("Specific power attenuation (km-1)")
#plt.xscale("log")
plt.ylabel("Altitude (km)")
#plt.title("Specific attenuation profiles at various frequencies")
plt.legend()
plt.grid(True)
#plt.tight_layout()
plt.savefig(out_dir / "specific_attenuation_profiles_pycraf.pdf", bbox_inches="tight")

pic latex for sigma_T fct of simulated pwv profile

In [63]:
from src.cosmo_lidar.mc_tools import hybrid_lin_geom
import numpy as np
import pycraf
from pycraf import conversions as cnv
from astropy import units as u
from scipy.integrate import trapezoid  # Pour l'intégration numérique
import matplotlib.pyplot as plt
def add_gaussian_layer(z, profile_base, center_alt, width, amplitude):
    """
    Ajoute une couche gaussienne au profil de base.
    z          : vecteur altitude (m)
    profile_base: profil P_water original (hPa)
    center_alt : altitude du centre de la couche (m)
    width      : écart-type (épaisseur) de la couche (m)
    amplitude  : intensité max de la couche ajoutée (hPa)
    """
    # Formule de la gaussienne
    gaussian = amplitude * np.exp(-0.5 * ((z - center_alt) / width)**2)
    
    # On ajoute au profil existant
    return profile_base + gaussian


# --- Paramètres fixes ---
pi = np.pi     
frequency = np.array([150e9]) 
theta_b = np.full(len(frequency), 1*60*pi/(2*180*3600))
elev = 90
N_MC = 70
epsilon = 0.622
R_water = 461.5
Z_obs = 5000 # Altitude de l'observateur (pour votre formule)
N_Z = 200


# --- Grille Haute Résolution (z_1) pour la physique ---
z_1 = np.linspace(5000, 15000, 2000) # m
z_1_km = (z_1 * u.m).to(u.km)

# Profils de base Pycraf
profile = pycraf.atm.profile_standard(z_1_km)
T_1 = profile[0].value
P_1 = profile[1].value
P_water_base = profile[3].value # Profil de forme de base
Tc = T_1 - 273.15
P_sat = 6.112 * np.exp((17.67 * Tc) / (Tc + 243.5))

# --- Grille Réduite (zg) pour le calcul Monte Carlo (fixe mais conservatrice) ---
# On prend un z_break un peu plus haut (4000 au lieu de 2000) pour accommoder les profils étirés
z_break_fixe = 9000 
N_lin = 4 * N_Z // 5
N_geom = N_Z - N_lin
zg = hybrid_lin_geom(5000, z_break_fixe, 20000, N_lin=N_lin, N_geom=N_geom, gamma=2)

# Interpolation du T et P sur la grille réduite (supposés constants, seule l'eau bouge)
T_g = np.interp(zg, z_1, T_1)
P_g = np.interp(zg, z_1, P_1)

In [67]:
import numpy as np
import matplotlib.pyplot as plt

# =============================================================================
# 1. PRÉPARATION : Limite Physique (Saturation)
# =============================================================================

# Formule de Magnus pour P_sat (hPa) en fonction de T (Celsius)
Tc = T_1 - 273.15
P_sat = 6.112 * np.exp((17.67 * Tc) / (Tc + 243.5))

# Conversion en densité max possible (g/m^3) pour le plot
rho_sat = (P_sat * 100) / (R_water * T_1) * 1000 

def add_gaussian_layer(z, profile_base, center_alt, width, amplitude):
    gaussian = amplitude * np.exp(-0.5 * ((z - center_alt) / width)**2)
    return profile_base + gaussian

# =============================================================================
# 2. CONTRÔLE VISUEL (Spaghetti Plot)
# =============================================================================

plt.figure(figsize=(10, 8))

# Plot du profil de base et de la limite de saturation
rho_base = (P_water_base * 100) / (R_water * T_1) * 1000
plt.plot(rho_base, z_1/1000, 'k-', linewidth=3, label='Base Pycraf')
plt.plot(rho_sat, z_1/1000, 'r--', linewidth=2, label='Saturation (Physics)')

print("Génération de 10 profils pour contrôle visuel...")

for k in range(10):
    # --- Tirage Aléatoire ---
    z_layer = np.random.uniform(5500, 7000)   # Altitude de la couche
    width_layer = np.random.uniform(200, 1000) # Épaisseur
    amp_layer = np.random.uniform(0.0, 2.5)    # Intensité (hPa)
    bg_scale = np.random.uniform(0.2, 1)     # Variation du fond
    
    # --- Construction ---
    P_base_scaled = P_water_base * bg_scale
    P_final = add_gaussian_layer(z_1, P_base_scaled, z_layer, width_layer, amp_layer)
    
    # --- Clipping Physique (Saturation) ---
    # On empêche l'eau de dépasser la saturation (ce qui créerait des nuages liquides, 
    # mais pour le délai on regarde surtout la vapeur)
    P_final = np.minimum(P_final, P_sat)
    
    # Conversion en densité
    rho_final = (P_final * 100) / (R_water * T_1) * 1000
    
    plt.plot(rho_final, z_1/1000, alpha=0.6, linewidth=1)

plt.xlabel(r"Density $\rho_{water}$ [g/m$^3$]")
plt.ylabel("Altitude [km]")
#plt.title("Contrôle : Profils avec perturbations Gaussiennes")
plt.legend()
plt.grid(True)
plt.xlim(0, 1.2) # Ajustez selon vos valeurs typiques
plt.ylim(5, 13)
plt.savefig(out_dir / "simulated_water_vapor_profile.pdf", bbox_inches="tight")

Génération de 10 profils pour contrôle visuel...


In [70]:
%store -r PWV_res
%store -r std_vert_res
%store -r z_moy_res
%store -r T_ant_res 
%store -r sigma_T_res

PWV_res = PWV_res
std_vert_res =std_vert_res
z_moy_res = z_moy_res
T_ant_res = T_ant_res
sigma_T_res = sigma_T_res

In [69]:
print (z_moy_res)

[6803.075828   6762.29219424 6624.86445336 6698.26712766 6539.57227697
 6421.0247046  6760.35069658 6907.58624272 6493.38527493 6851.8018523
 6742.43124727 6857.93958757 6683.9832493  6572.35104139 6519.26458748
 6513.89070275 6615.87838875 6734.47860298 6874.86921102 6772.00527729
 6419.43665005 6791.0167451  6678.06651164 6673.51813146 6591.80478047
 6474.43534124 6263.12947294 6530.61689666 6810.98994591 6651.05738622
 6905.56311793 6495.873703   6588.50862449 6099.95977072 6393.15086067
 6182.57858111 6951.46013621 6891.0964996  6723.06736018 6511.49838146
 6555.59971003 6590.54214243 6846.77548483 6823.11839328 6429.45450233
 6134.41138936 6436.41042493 6914.453749   6459.06165506 6641.92240087
 6337.40750021 6415.7409825  6697.66196329 6673.90890977 6286.23554834
 6521.65004048 6929.54362094 6552.9325328  6537.74774775 6178.08624311
 6814.11356587 6391.61732574 6834.29432952 6875.63670898 6447.94726662
 6450.80291574 6414.1224443  6266.72448224 6873.03207903 6876.33017249
 6578.4

In [72]:
plt.figure(figsize=(10, 6))
plt.scatter(PWV_res/1000, sigma_T_res*1000, s=40, alpha=0.8)


#plt.title(f"Vérification de la relation affine (N={N_scenarios})")
plt.xlabel("PWV")
plt.ylabel(r"$\sigma_T$ [mK]")
#plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.savefig(out_dir / "sigma_t_vs_pwv_simulated.pdf", bbox_inches="tight")

In [112]:
dist_term = (z_moy_res - Z_obs)**2
geom_factor = std_vert_res**2 + dist_term

# Le facteur X pour le plot
X_factor = np.sqrt(PWV_res * geom_factor)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(np.sqrt(PWV_res/1000), sigma_T_res*1000, c=((z_moy_res/1000-Z_obs/1000)**2+(std_vert_res/1000)**2), cmap='rainbow', s=40, alpha=0.8)
plt.colorbar(label=r"$\sigma_{vert}^2 + (\bar{z} - z_{min})^2 (km^2)$ ")

# Régression linéaire pour voir si c'est affine
coeffs = np.polyfit(X_factor, sigma_T_res, 1)
poly = np.poly1d(coeffs)
#plt.plot(X_factor, poly(X_factor), 'r--', alpha=0.5, label=f'Fit: y={coeffs[0]:.2e}x + {coeffs[1]:.2e}')

#plt.title(f"Vérification de la relation affine (N={N_scenarios})")
plt.xlabel(r"$\sqrt{PWV}$ (mm$^{1/2}$)")
plt.ylabel(r"$\sigma_T$ [mK]")
#plt.legend()
plt.grid(True, alpha=0.6)
plt.savefig(out_dir / "sigma_t_vs_sqrtpwv_z_moy_std__simulated.pdf", bbox_inches="tight")


In [114]:
dist_term = (z_moy_res - Z_obs)**2
geom_factor = std_vert_res**2 + dist_term

# Le facteur X pour le plot
X_factor = np.sqrt(PWV_res * geom_factor)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(np.sqrt(PWV_res/1000), sigma_T_res*1000, c=(z_moy_res/1000-Z_obs/1000)**2, cmap='rainbow', s=40, alpha=0.8)
plt.colorbar(label=r"$ (\bar{z} - z_{min})^2 (km^2)$ ")

# Régression linéaire pour voir si c'est affine
coeffs = np.polyfit(X_factor, sigma_T_res, 1)
poly = np.poly1d(coeffs)
#plt.plot(X_factor, poly(X_factor), 'r--', alpha=0.5, label=f'Fit: y={coeffs[0]:.2e}x + {coeffs[1]:.2e}')

#plt.title(f"Vérification de la relation affine (N={N_scenarios})")
plt.xlabel(r"$\sqrt{PWV}$ (mm$^{1/2}$)")
plt.ylabel(r"$\sigma_T$ [mK]")
#plt.legend()
plt.grid(True, alpha=0.6)
plt.savefig(out_dir / "test_1.pdf", bbox_inches="tight")

In [115]:
dist_term = (z_moy_res - Z_obs)**2
geom_factor = std_vert_res**2 + dist_term

# Le facteur X pour le plot
X_factor = np.sqrt(PWV_res * geom_factor)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(np.sqrt(PWV_res/1000), sigma_T_res*1000, c=((std_vert_res/1000)**2), cmap='rainbow', s=40, alpha=0.8)
plt.colorbar(label=r"$\sigma_{vert}^2 (km^2)$ ")

# Régression linéaire pour voir si c'est affine
coeffs = np.polyfit(X_factor, sigma_T_res, 1)
poly = np.poly1d(coeffs)
#plt.plot(X_factor, poly(X_factor), 'r--', alpha=0.5, label=f'Fit: y={coeffs[0]:.2e}x + {coeffs[1]:.2e}')

#plt.title(f"Vérification de la relation affine (N={N_scenarios})")
plt.xlabel(r"$\sqrt{PWV}$ (mm$^{1/2}$)")
plt.ylabel(r"$\sigma_T$ [mK]")
#plt.legend()
plt.grid(True, alpha=0.6)
plt.savefig(out_dir / "test_2.pdf", bbox_inches="tight")

In [78]:
plt.figure(figsize=(7, 5))
plt.scatter(z_moy_res - Z_obs, std_vert_res, s=40, alpha=0.8)


#plt.title(f"Vérification de la relation affine (N={N_scenarios})")
plt.xlabel(r"$\bar{z} - z_{min}$ (m)")
plt.ylabel(r"$\sigma_{vert}$ (m)")
#plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.savefig(out_dir / "z_moy_vs_std_simulated.pdf", bbox_inches="tight")

In [ ]:
dist_term = (z_moy_res - Z_obs)**2
geom_factor = std_vert_res**2 + dist_term

# Le facteur X pour le plot
X_factor = np.sqrt(PWV_res /1000 * geom_factor)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(np.sqrt(PWV_res/1000), sigma_T_res*1000, c=((z_moy_res-Z_obs)**2+std_vert_res**2), cmap='viridis', s=40, alpha=0.8)
plt.colorbar(label='(z_moy-5000)^2 + std_vert^2 [m2]')

# Régression linéaire pour voir si c'est affine
coeffs = np.polyfit(X_factor, sigma_T_res, 1)
poly = np.poly1d(coeffs)
#plt.plot(X_factor, poly(X_factor), 'r--', alpha=0.5, label=f'Fit: y={coeffs[0]:.2e}x + {coeffs[1]:.2e}')

plt.title(f"Vérification de la relation affine (N={N_scenarios})")
plt.xlabel(r"Facteur $\sqrt{PWV}$")
plt.ylabel(r"$\sigma_T$ [mK]")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

# Affichage des plages de variations pour confirmer la diversité
print(f"Variation z_moy : {z_moy_res.min():.0f} m à {z_moy_res.max():.0f} m")
print(f"Variation std_vert : {std_vert_res.min():.0f} m à {std_vert_res.max():.0f} m")

In [113]:
plt.figure(figsize=(10, 6))
X_factor = np.sqrt(PWV_res /1000 * geom_factor)
plt.scatter(X_factor/1000, sigma_T_res*1000, s=40, alpha=0.8)
#plt.colorbar(label='(z_moy-5000)^2 + std_vert^2 [m2]')

# Régression linéaire pour voir si c'est affine
coeffs = np.polyfit(X_factor/1000, sigma_T_res*1000, 1)
poly = np.poly1d(coeffs)
plt.plot(X_factor/1000, poly(X_factor/1000), 'r--', alpha=0.5, label=f'Fit: y={coeffs[0]:.2e}x + {coeffs[1]:.2e}')

#plt.title(f"Vérification de la relation affine (N={N_scenarios})")
plt.xlabel(r"$\sqrt{PWV \cdot (\sigma_{vert}^2 + (\bar{z} - z_{min})^2)}$ (kg$^{1/2}\cdot km$)")
plt.ylabel(r"$\sigma_T$ [mK]")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.savefig(out_dir / "sigma_t_vs_factor_simulated.pdf", bbox_inches="tight")

In [85]:
# 1. Préparer les données de couleur pour avoir une échelle commune
# Cela garantit que le "jaune" sur le graphe de gauche vaut la même chose que sur celui de droite
c_values = z_moy_res - Z_obs
vmin = np.min(c_values)
vmax = np.max(c_values)

# 2. Création de la figure
# constrained_layout=True aide à gérer l'espace automatiquement avec la colorbar commune
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

# --- Graphe GAUCHE ---
# On ajoute vmin=vmin, vmax=vmax
sc1 = ax1.scatter(PWV_res/1000, T_ant_res, 
                  c=c_values, cmap='viridis', s=40, alpha=0.8,
                  vmin=vmin, vmax=vmax)

ax1.set_xlabel("PWV [mm]")
ax1.set_ylabel(r"$T_{ant}$ [K]")
ax1.grid(True, linestyle=':', alpha=0.6)
#ax1.set_title(r"$T_{ant}$ vs PWV")

# --- Graphe DROITE ---
# On utilise la même normalisation vmin/vmax
sc2 = ax2.scatter(PWV_res/1000, (T_ant_res/PWV_res)*1000, 
                  c=c_values, cmap='viridis', s=40, alpha=0.8,
                  vmin=vmin, vmax=vmax)

ax2.set_xlabel("PWV [mm]")
ax2.set_ylabel(r"$T_{ant} / \mathrm{PWV}$ [K/mm]")
ax2.set_ylim(5.5, 8)
ax2.grid(True, linestyle=':', alpha=0.6)
#ax2.set_title(r"Opacité spécifique ($T_{ant}/\mathrm{PWV}$)")

# --- BARRE DE COULEUR UNIQUE ---
# On attache la colorbar à la figure en lui disant de prendre de la place sur ax1 et ax2
cbar = fig.colorbar(sc1, ax=[ax1, ax2], aspect=30)
cbar.set_label(r'$\bar{z} - z_{min}$ [m]')

plt.savefig(out_dir / "t_ant_vs_pwv_zmoy_simulated.pdf", bbox_inches="tight")

In [87]:
# 1. Préparer les données de couleur pour avoir une échelle commune
# Cela garantit que le "jaune" sur le graphe de gauche vaut la même chose que sur celui de droite
c_values = std_vert_res
vmin = np.min(c_values)
vmax = np.max(c_values)

# 2. Création de la figure
# constrained_layout=True aide à gérer l'espace automatiquement avec la colorbar commune
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

# --- Graphe GAUCHE ---
# On ajoute vmin=vmin, vmax=vmax
sc1 = ax1.scatter(PWV_res/1000, T_ant_res, 
                  c=c_values, cmap='viridis', s=40, alpha=0.8,
                  vmin=vmin, vmax=vmax)

ax1.set_xlabel("PWV [mm]")
ax1.set_ylabel(r"$T_{ant}$ [K]")
ax1.grid(True, linestyle=':', alpha=0.6)
#ax1.set_title(r"$T_{ant}$ vs PWV")

# --- Graphe DROITE ---
# On utilise la même normalisation vmin/vmax
sc2 = ax2.scatter(PWV_res/1000, (T_ant_res/PWV_res)*1000, 
                  c=c_values, cmap='viridis', s=40, alpha=0.8,
                  vmin=vmin, vmax=vmax)

ax2.set_xlabel("PWV [mm]")
ax2.set_ylabel(r"$T_{ant} / \mathrm{PWV}$ [K/mm]")
ax2.set_ylim(5.5, 8)
ax2.grid(True, linestyle=':', alpha=0.6)
#ax2.set_title(r"Opacité spécifique ($T_{ant}/\mathrm{PWV}$)")

# --- BARRE DE COULEUR UNIQUE ---
# On attache la colorbar à la figure en lui disant de prendre de la place sur ax1 et ax2
cbar = fig.colorbar(sc1, ax=[ax1, ax2], aspect=30)
cbar.set_label(r'$\sigma_{vert}$ [m]')

plt.savefig(out_dir / "t_ant_vs_pwv_std_vert_simulated.pdf", bbox_inches="tight")

In [104]:
%store -r z_max_list
%store -r T_list
%store -r sigma_list
%store -r T_wo_w_list

z_max_list = z_max_list
T_list = T_list
sigma_list = sigma_list
T_wo_w_list = T_wo_w_list

In [107]:
plt.figure()
plt.plot(z_max_list/1000, T_list, marker= 'o', label="T_ant")
#plt.plot(z_max_list, sigma_list*1000, marker = 'o',label="sigma_T")
#plt.axvline(z_max_opt, linestyle="--", label=f"z_max_opt = {z_max_opt:.1f} m")
plt.xlabel(r"$z_{\mathrm{max}}$ [km]")
plt.ylabel(r"$T_{\mathrm{ant}}$ [K]")
#plt.legend()
plt.grid(True, which="both")
plt.savefig(out_dir / "t_ant_vs_z_max_pycraf.pdf", bbox_inches="tight")

In [106]:
plt.figure()
plt.plot(z_max_list/1000, T_list- T_wo_w_list, marker= 'o', label="only water")
plt.plot(z_max_list/1000, T_list, marker= 'o', label="water + dry air")
#plt.plot(z_max_list, sigma_list*1000, marker = 'o',label="sigma_T")
#plt.axvline(z_max_opt, linestyle="--", label=f"z_max_opt = {z_max_opt:.1f} m")
plt.xlabel(r"$z_{\mathrm{max}}$ [km]")
plt.ylabel(r"$T_{\mathrm{ant}}$ [K]")
plt.legend()
plt.grid(True, which="both")
plt.savefig(out_dir / "t_ant_water_and_wo_water_vs_z_max_pycraf.pdf", bbox_inches="tight")

Calcul un peu nul sur sigma_T avec variation de PWV

In [108]:
import numpy as np
import pycraf
from pycraf import conversions as cnv
from astropy import units as u
from scipy.integrate import trapezoid  # Pour l'intégration numérique
import matplotlib.pyplot as plt


# =============================================================================
# 2. INITIALISATION ET MAILLAGE HAUTE RÉSOLUTION (z_1)
# =============================================================================

# Paramètres généraux
pi = np.pi     
frequency = np.array([150e9]) # Hz
theta_b = np.full(len(frequency), 1*60*pi/(2*180*3600))
elev = 90
N_MC = 100
epsilon = 0.622
R_water = 461.5 # J/kg/K

# -- 2a. Création de la grille fine linéaire (z_1) --
# On prend large (ex: 2000 points) pour avoir une bonne précision sur z_moy et z_break
altitudes_hr = np.linspace(5000, 15000, 2000) # m
z_1 = altitudes_hr
z_1_km = (z_1 * u.m).to(u.km)

# -- 2b. Récupération des profils Pycraf sur la grille fine --
profile = pycraf.atm.profile_standard(z_1_km)
T_1 = profile[0].value        # K
P_1 = profile[1].value        # hPa
rho_water_1_base = profile[2].value # g/m^3 (Profil standard de base)
P_water_1_base = profile[3].value   # hPa (Profil standard de base)

# =============================================================================
# 3. CRÉATION DE LA GRILLE RÉDUITE OPTIMISÉE (zg)
# =============================================================================

N_Z = 150
N_lin = 4 * N_Z // 5
N_geom = N_Z - N_lin
gamma = 2
Z_FLOOR = 5000

# Calcul du z_break sur le profil standard (la forme ne change pas avec le scaling)
z_break = calcul_z_percentile_wvc(z_1, rho_water_1_base, 90)
print(f"Altitude de coupure (90% eau) : {z_break:.1f} m")

zmin = max(float(np.nanmin(z_1)), Z_FLOOR)
zmax = float(np.nanmax(z_1))

# Génération du maillage réduit
zg = hybrid_lin_geom(zmin, z_break, zmax, N_lin=N_lin, N_geom=N_geom, gamma=gamma)

# =============================================================================
# 4. BOUCLE DE CALCUL AVEC SCALING PWV
# =============================================================================

# Facteurs d'échelle PWV
x_scale = np.linspace(0.01, 10, 50)

# Tableaux pour stocker les résultats
sigma_T_results = np.zeros_like(x_scale)
T_ant_results = np.zeros_like(x_scale)
z_moy_results = np.zeros_like(x_scale)
std_vert_results = np.zeros_like(x_scale)
PWV_results = np.zeros_like(x_scale) # Pour vérification

print(f"Lancement de la simulation pour {len(x_scale)} points...")

for i, val in enumerate(x_scale):
    
    # --- A. Mise à l'échelle sur la grille HAUTE RÉSOLUTION (z_1) ---
    # On travaille sur z_1 pour calculer les intégrales (z_moy, std) avec précision
    
    P_water_1_scaled = P_water_1_base * val
    
    # Recalcul de rho_water sur z_1 (Loi des gaz parfaits pour la vapeur d'eau)
    # rho = P / (R * T). Attention aux unités : P en Pascal pour formule SI
    # P_water_1_scaled est en hPa -> * 100 pour Pa
    # T_1 en K
    # R_water = 461.5 J/kg/K
    # Resultat en kg/m3 -> * 1000 pour g/m3
    rho_water_1_scaled = (P_water_1_scaled * 100) / (R_water * T_1) * 1000 # g/m^3
    
    # --- B. Calcul des statistiques verticales sur z_1 ---
    
    # Calcul du PWV total (intégrale de rho selon z) en g/m^2 ou mm
    current_PWV = trapezoid(rho_water_1_scaled, z_1)
    
    # Altitude moyenne (z_moy)
    # Intégrale (rho * z) / PWV
    z_moy = trapezoid(rho_water_1_scaled * z_1, z_1) / current_PWV
    
    # Variance et Ecart-type vertical
    var_vert = trapezoid(rho_water_1_scaled * (z_1 - z_moy)**2, z_1) / current_PWV
    std_vert = np.sqrt(var_vert)
    
    # Stockage des stats
    z_moy_results[i] = z_moy
    std_vert_results[i] = std_vert
    PWV_results[i] = current_PWV / 1000 # Conversion souvent utile (kg/m^2) si rho en g
    
    # --- C. Interpolation vers la grille RÉDUITE (zg) pour Monte Carlo ---
    
    # Interpolation des champs scalés depuis z_1 vers zg
    T_g = np.interp(zg, z_1, T_1)
    P_g = np.interp(zg, z_1, P_1)
    P_water_g = np.interp(zg, z_1, P_water_1_scaled)
    
    # Recalcul du WVMR sur la grille réduite
    # WVMR = epsilon * P_w / (P_dry) = epsilon * P_w / (P_tot - P_w)
    denom = P_g - P_water_g
    denom[denom <= 0] = 1e-6 # Sécurité numérique
    
    WVMR_g = epsilon * P_water_g / denom * 1000 # g/kg
    
    # --- D. Appel de la fonction de prédiction ---
    
    # On passe les vecteurs réduits (_g)
    sig, temp = predict_SNR_T(
        frequency=frequency,
        theta_b=theta_b,
        z=zg,           # Grille réduite
        WVMR=WVMR_g,    # Profil interpolé
        elev=elev,
        T=T_g,
        P=P_g,
        N_MC=N_MC
    )
    
    # Moyenne si la fonction retourne un array
    sigma_T_results[i] = np.mean(sig)
    T_ant_results[i] = np.mean(temp)

Altitude de coupure (90% eau) : 9487.4 m
Lancement de la simulation pour 50 points...


In [ ]:
plt.figure()
plt.scatter(np.sqrt(PWV_results), sigma_T_results*1000)

plt.xlabel("sqrt(PWV) ()")
plt.ylabel("sigma_T_ant (mK)")

plt.grid(True, which="both")


plt.figure()
plt.scatter(PWV_results, sigma_T_results*1000/np.sqrt(PWV_results))

plt.xlabel("sqrt(PWV) ()")
plt.ylabel("sigma_T_ant (mK)")

plt.grid(True, which="both")

In [111]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Préparation des données ---
x_val = np.sqrt(PWV_results)
y_val = sigma_T_results * 1000

# --- 2. Calcul de la régression linéaire ---
# np.polyfit(x, y, 1) retourne [pente, ordonnée_à_l_origine]
coeffs = np.polyfit(x_val, y_val, 1)
poly = np.poly1d(coeffs)

# (Optionnel) Calcul du coefficient de corrélation R^2 pour la légende
correlation_matrix = np.corrcoef(x_val, y_val)
correlation_xy = correlation_matrix[0, 1]
r_squared = correlation_xy**2

# --- 3. Tracé de la figure unique ---
fig, ax = plt.subplots(figsize=(8, 6))

# Nuage de points
ax.scatter(x_val, y_val, edgecolors='none', label='Simulated data')

# Droite de régression
# On trace la ligne rouge pointillée par-dessus
# On utilise x_val trié pour que la ligne soit tracée proprement
x_range = np.linspace(min(x_val), max(x_val), 100)
ax.plot(x_range, poly(x_range), 'r--', 
        label=r'Fit : $y = %.2f x + %.2f$ ($R^2=%.3f$)' % (coeffs[0], coeffs[1], r_squared))

# --- 4. Mise en forme LaTeX ---
ax.set_xlabel(r"$\sqrt{\mathrm{PWV}} \ (\mathrm{mm}^{1/2})$")
ax.set_ylabel(r"$\sigma_{T} \ (\mathrm{mK})$")

ax.grid(True)
ax.legend(loc='best') # Affiche la légende avec l'équation

# Sauvegarde
plt.tight_layout()


plt.savefig(out_dir / "sigma_T_various_pwv_same_wvp.pdf", bbox_inches="tight")

Multifrequencies part, we do the graph for latex

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pycraf.atm
from astropy import units as u

# --- 1. Initialisation des paramètres (Données fournies) ---
pi = np.pi  
f_93 = 93e9   
f_120 = 120e9 
f_145 = 145e9
f_150 = 150e9 
f_179 = 179e9 
f_188 = 188e9 
f_225 = 225e9
f_280 = 280e9

frequencies = np.array([f_93, f_120, f_150, f_188, f_280]) # Array de 4 fréquences

theta_b = np.full(len(frequencies), 1*60*pi/(2*180*3600))
elevation = 45

# Profil atmosphérique de base
altitudes = np.geomspace(1, 15000, 200) + 4999 # m
altitudes_km = (altitudes * u.m).to(u.km)
Temperature = pycraf.atm.profile_standard(altitudes_km)[0].value
Pressure = pycraf.atm.profile_standard(altitudes_km)[1].value
P_water_base = pycraf.atm.profile_standard(altitudes_km)[3].value # Renommé pour clarté

# --- 2. Boucle de calcul Multi-Fréquences ---

PWV_base = 1.22
x = np.linspace(0.01, 10, 100) # Facteurs d'échelle
PWV_x = x * PWV_base           # Axe X (PWV en mm)

# Initialisation de la matrice de résultats
# Dimensions : [nombre de points PWV, nombre de fréquences]
T_ant_results = np.zeros((len(x), len(frequencies)))

print("Calcul en cours...")

for i, val in enumerate(x):
    
    # Mise à l'échelle de la vapeur d'eau
    P_water_test = P_water_base * val
    
    # Appel de la fonction (qui renvoie un array de taille 4)
    # On stocke le résultat dans la ligne 'i' de la matrice
    T_ant_results[i, :] = Calcul_T_ant_1_el(
        frequencies, 
        theta_b, 
        altitudes, 
        Temperature, 
        Pressure, 
        P_water_test, 
        elevation
    )

# --- 3. Tracé des courbes ---

plt.figure(figsize=(10, 6))

# On boucle sur chaque fréquence pour tracer sa courbe correspondante
for j, freq in enumerate(frequencies):
    freq_ghz = freq / 1e9
    # On trace PWV vs la colonne j de la matrice de résultats
    plt.plot(PWV_x, T_ant_results[:, j], label=f'{freq_ghz:.0f} GHz')

plt.xlabel(r"$\mathrm{PWV} \ (\mathrm{mm})$")
plt.ylabel(r"$T_{\mathrm{ant}}$ [K]")
#plt.title(r"Évolution de $T_{\mathrm{ant}}$ selon le contenu en eau (Multi-Fréquences)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.savefig(out_dir / "t_ant_various_frequencies_function_pwv.pdf", bbox_inches="tight")

Calcul en cours...


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit



# 1. Définition du modèle (Saturation exponentielle)
def modele_Tant(pwv, k, c):
    # k : Température asymptotique (proche de T_physique moyenne)
    # c : coefficient lié à l'absorption massique de l'eau
    return k * (1 - np.exp(-c * pwv))

# 2. Boucle sur les fréquences
# On prépare des listes pour stocker les résultats si besoin
k_opt_list = []
c_opt_list = []

print(f"{'Fréquence (GHz)':<20} | {'k (K)':<10} | {'c (coeff)':<10}")
print("-" * 50)

plt.figure(figsize=(10, 6))

for j, freq in enumerate(frequencies):
    if freq == f_120 :
        continue
    freq_ghz = freq / 1e9
    
    # --- A. Sélection des données pour la fréquence j ---
    # x_data est toujours le même (PWV)
    # y_data change : on prend la colonne j
    y_data_freq = T_ant_results[:, j]
    
    # --- B. Le "Fit" (Ajustement) ---
    # p0=[280, 0.1] sont des valeurs initiales pour aider l'algo
    # k ~ 280 K (température ambiante), c ~ 0.1 (absorption)
    try:
        popt, pcov = curve_fit(modele_Tant, PWV_x, y_data_freq, p0=[280, 0.1])
        
        k_val = popt[0]
        c_val = popt[1]
        
        # Stockage
        k_opt_list.append(k_val)
        c_opt_list.append(c_val)
        
        # Affichage console
        print(f"{freq_ghz:<20.0f} | {k_val:<10.2f} | {c_val:<10.4f}")
        
        # --- C. Tracé pour vérification ---
        # On trace les points bruts
        plt.scatter(PWV_x, y_data_freq, alpha=0.3, label=f'Data' if j==0 else "")
        
        # On trace la courbe modèle
        # On crée une ligne lisse pour le modèle
        x_smooth = np.linspace(PWV_x.min(), PWV_x.max(), 200)
        y_model = modele_Tant(x_smooth, k_val, c_val)
        
        #color = plt.gca().lines[-1].get_color() # Récupère la couleur du scatter précédent (astuce matplotlib)
        # Note: scatter ne met pas à jour lines, donc on force la couleur via le cycle ou on laisse auto
        plt.plot(x_smooth, y_model, '--', linewidth=2, label=f'Fit {freq_ghz:.0f} GHz (c={c_val:.3f})')
        
    except RuntimeError:
        print(f"Erreur de fit pour {freq_ghz} GHz")

plt.xlabel(r"$\mathrm{PWV} \ (\mathrm{mm})$")
plt.ylabel(r"$T_{\mathrm{ant}}$ [K]")
#plt.title(r"Ajustement du modèle $T = k(1 - e^{-c \cdot PWV})$")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
#plt.show()
plt.savefig(out_dir / "t_ant_fit_beer_lambert_pwv.pdf", bbox_inches="tight")

# Conversion en arrays numpy pour usage ultérieur
k_opts = np.array(k_opt_list)
c_opts = np.array(c_opt_list)




Fréquence (GHz)      | k (K)      | c (coeff) 
--------------------------------------------------
93                   | 85.94      | 0.0442    
150                  | 5590.12    | 0.0013    
188                  | 252.07     | 0.3625    
280                  | 318.05     | 0.0812    


In [8]:
import numpy as np
import pycraf
from pycraf import conversions as cnv
from astropy import units as u
import matplotlib.pyplot as plt
from src.cosmo_lidar.mc_tools import calcul_snr, scale_snr_for_variable_bins, Monte_Carlo_T_ant_profile, generate_Pwater_MC_lognormal

# --- 1. Paramètres initiaux ---
pi = np.pi     

# Définition des multiples fréquences

frequencies = np.array([f_93, f_120, f_150, f_188, f_280]) # Hz

# Theta_b pour chaque fréquence (ici supposé constant à 1 arcmin, 
# mais vous pourriez le faire varier en fct de lambda/D si besoin)
theta_b_all = np.full(len(frequencies), 1*60*pi/(2*180*3600))


elev = 45 # en deg
N = 500
N_MC = 200

# --- 2. Profils Atmosphériques (Commun à toutes les fréquences) ---
altitudes = np.geomspace(1, 30000, 1000) # m
altitudes = altitudes + 4999 # m

altitudes_km = altitudes * u.m       
altitudes_km = altitudes_km.to(u.km) 

# Récupération des données Pycraf
Temperature = pycraf.atm.profile_standard(altitudes_km)[0].value # K
Pressure = pycraf.atm.profile_standard(altitudes_km)[1].value # hPa
rho_water = pycraf.atm.profile_standard(altitudes_km)[2].value # g/m3
P_water = pycraf.atm.profile_standard(altitudes_km)[3].value # hPa

# Calcul WVMR
epsilon = 0.622 
WVMR = epsilon * P_water / (Pressure - P_water) # kg/kg
WVMR = WVMR * 1000 # g/kg

# --- 3. Calcul du SNR (Commun car dépend du Lidar, pas du radiomètre) ---
# Valeurs de Patrick
A = 106533.049
c = 1.470303e-05

SNR_no_scale = calcul_snr(A, c, WVMR, altitudes, Pressure, Temperature, elev)
# On prend l'index [0] si scale_snr renvoie un tuple ou array
SNR = scale_snr_for_variable_bins(altitudes, SNR_no_scale, elev=90, dz_ref=30.0)[0]


# --- 4. Boucle de Simulation par Fréquence ---

# Dictionnaire pour stocker les résultats : {frequence: (sigma_prof, T_mean_prof, T_dry_prof)}
results_dict = {}

print(f"Lancement des simulations Monte Carlo pour {len(frequencies)} fréquences...")

for i, freq in enumerate(frequencies):
    freq_ghz = freq / 1e9
    print(f"Processing {freq_ghz:.0f} GHz...")
    
    # On prépare les arguments scalaires ou tableaux unitaires pour la fonction
    f_input = np.array([freq])
    theta_b_input = np.array([theta_b_all[i]])
    
    # Appel de la fonction MC
    # Renvoie : sigma_prof, T_mean_prof, T_dry_prof
    simu_freq = Monte_Carlo_T_ant_profile(
        f_input, 
        theta_b_input, 
        N, 
        elev, 
        generate_Pwater_MC_lognormal, 
        N_MC, 
        WVMR, 
        SNR, 
        Temperature, 
        Pressure, 
        altitudes
    )
    
    # Stockage
    results_dict[freq] = simu_freq



Lancement des simulations Monte Carlo pour 5 fréquences...
Processing 93 GHz...
Processing 120 GHz...
Processing 150 GHz...
Processing 188 GHz...
Processing 280 GHz...


In [9]:
# --- 5. Exemple de visualisation (Comparaison des fréquences) ---
plt.figure(figsize=(10, 6))

for freq in frequencies:
    freq_ghz = freq / 1e9
    # Récupération des résultats
    sigma_prof, T_mean_prof, T_dry_prof = results_dict[freq]
    
    # On trace T_mean (Total)
    plt.plot(altitudes/1000, T_mean_prof, label=f'{freq_ghz:.0f} GHz')
    
    # Optionnel : afficher sigma en pointillé (x10 pour visibilité)
    # plt.plot(altitudes/1000, sigma_prof*10, '--', alpha=0.5)

plt.xlabel(r"Height integration $z_{max}$ [km]")
plt.ylabel(r"T_{ant} [K]")
#plt.title(r"Profils verticaux $T_{ant}$ pour différentes fréquences")
plt.xlim(5,20)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(out_dir / "t_ant_zmax_various_frequencies.pdf", bbox_inches="tight")


In [11]:
import matplotlib.pyplot as plt
import numpy as np
altitudes = np.geomspace(1, 30000, 1000) # m
altitudes = altitudes + 4999 # m
# Préparation de l'axe X en km
alt_km = altitudes / 1000

# =============================================================================
# Graphique 1 : T_ant Normalisée (T(z) / T_final)
# =============================================================================
plt.figure(figsize=(10, 6))
frequencies = np.array([f_120,f_150, f_179, f_188])
for freq in sorted(frequencies):
    freq_ghz = freq / 1e9
    sigma_prof, T_mean_prof, _ = results_dict[freq]
    
    # Valeur asymptotique (z_infini)
    T_inf = T_mean_prof[-1]
    
    # Normalisation
    # Attention à la division par 0 si T_inf est nul (peu probable physiquement ici)
    T_norm = T_mean_prof / T_inf if T_inf != 0 else np.zeros_like(T_mean_prof)
    
    plt.plot(alt_km, T_norm, label=f'{freq_ghz:.0f} GHz', linewidth=2)

plt.xlabel(r"Height integration $z_{max}$ [km]")
plt.ylabel(r"$T_{\mathrm{ant}}(z_{max}) \ / \ T_{\mathrm{ant}}(z_{\infty})$")
#plt.title(r"Saturation normalisée de la Température d'Antenne")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.xlim(5, 20) # Zoom sur la partie troposphérique pertinente
plt.tight_layout()
plt.savefig(out_dir / "t_ant_normalized_zmax_various_frequencies.pdf", bbox_inches="tight")


# =============================================================================
# Graphique 2 : Sigma_T Absolu en fonction de z
# =============================================================================
plt.figure(figsize=(10, 6))

for freq in sorted(frequencies):
    freq_ghz = freq / 1e9
    sigma_prof, _, _ = results_dict[freq]
    
    # On trace en Kelvin (ou multipliez par 1000 pour des mK)
    plt.plot(alt_km, sigma_prof, label=f'{freq_ghz:.0f} GHz', linewidth=2)

plt.xlabel(r"Height integration $z_{max}$ [km]")
plt.ylabel(r"$\sigma_T(z_{max})$ [K]")
#plt.title(r"Évolution cumulée de l'incertitude $\sigma_T$")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.xlim(5, 20)
plt.savefig(out_dir / "sigma_t_zmax_various_frequencies.pdf", bbox_inches="tight")


# =============================================================================
# Graphique 3 : Sigma_T Normalisé (Sigma(z) / Sigma_final)
# =============================================================================
plt.figure(figsize=(10, 6))

for freq in sorted(frequencies):
    freq_ghz = freq / 1e9
    sigma_prof, _, _ = results_dict[freq]
    
    # Valeur asymptotique
    sigma_inf = sigma_prof[-1]
    
    # Normalisation
    if sigma_inf > 0:
        sigma_norm = sigma_prof / sigma_inf
    else:
        sigma_norm = np.zeros_like(sigma_prof)
    
    plt.plot(alt_km, sigma_norm, label=f'{freq_ghz:.0f} GHz', linewidth=2)

plt.xlabel(r"Height integration $z_{max}$ [km]")
plt.ylabel(r"$\sigma_T(z_{max}) \ / \ \sigma_T(z_{\infty})$")
#plt.title(r"Saturation normalisée de $\sigma_T$")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.xlim(5, 20)
plt.savefig(out_dir / "sigma_t_normalized_zmax_various_frequencies.pdf", bbox_inches="tight")

KeyError: np.float64(179000000000.0)

In [12]:
print(sigma_prof, T_mean_prof)

[0.00000000e+00 4.74730548e-14 3.34185859e-13 1.10024835e-12
 2.80724152e-12 5.88182824e-12 1.05480785e-11 1.82944888e-11
 2.71828789e-11 3.90327444e-11 5.50657586e-11 7.35284354e-11
 1.04142900e-10 1.37255776e-10 1.70572715e-10 2.12731927e-10
 2.58419676e-10 3.11686912e-10 3.76050277e-10 4.40729541e-10
 5.44818914e-10 6.75802707e-10 7.95566048e-10 8.92266206e-10
 1.01153635e-09 1.16592198e-09 1.31727175e-09 1.52395636e-09
 1.80982443e-09 2.05388575e-09 2.27779225e-09 2.56049401e-09
 2.85854270e-09 3.18504887e-09 3.57394770e-09 4.01424993e-09
 4.34424626e-09 4.50985355e-09 4.84400725e-09 5.23719236e-09
 5.65007868e-09 6.22128845e-09 6.99891422e-09 7.69709570e-09
 8.53258352e-09 9.38124702e-09 9.81810680e-09 1.02423126e-08
 1.08708451e-08 1.19568502e-08 1.31098533e-08 1.37306454e-08
 1.41130550e-08 1.49644457e-08 1.61696366e-08 1.71775832e-08
 1.82462308e-08 1.93806873e-08 2.11840357e-08 2.32379132e-08
 2.49685791e-08 2.65534234e-08 2.75032000e-08 2.83327189e-08
 2.90601363e-08 3.042535